## PCA Analysis

In [27]:
%matplotlib inline
import yfinance as yf
import pandas as pd
import numpy as np
from matplotlib import pyplot as plt
import seaborn as sns
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from scipy.stats import ttest_ind, f_oneway

from sklearn.ensemble import GradientBoostingRegressor
from sklearn.metrics import mean_squared_error, r2_score

# Import Ensemble models
from sklearn.ensemble import RandomForestRegressor
from sklearn.ensemble import RandomForestClassifier
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.ensemble import ExtraTreesRegressor
from sklearn.ensemble import AdaBoostRegressor
from xgboost import XGBRegressor 

from sklearn.metrics import classification_report, accuracy_score
import cvxpy as cp
import re
import os
from sklearn.cluster import DBSCAN
from sklearn.manifold import TSNE
from matplotlib import cm
from sklearn.metrics import precision_score

from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC

from statsmodels.tsa.stattools import ccf
from grid_chart_util import create_stock_price_grid
from spark_init import initialize_spark
from config_loader import load_config
from pandas_util import convert_to_yfinance_format
from finance_util import calculate_technical_indicators
from pyspark.sql import SparkSession
from pyspark.sql import DataFrame as SparkDataFrame
from pyspark.sql.functions import col,dayofweek,current_date,date_sub,to_date,abs,lit,desc
from pyspark.sql.functions import sum as spark_sum,max as spark_max, min as spark_min
from pyspark.sql.types import IntegerType
from datetime import datetime, timedelta

In [28]:
# Get date threshold (last 3 weeks)
# date_threshold = datetime.today() - timedelta(weeks=26)
# Get today's date and replace the month and day to January 1st
# date_threshold = datetime.today().replace(month=1, day=1)
# Get today's date and calculate December 1 of the previous year
date_threshold = datetime(datetime.today().year - 1, 12, 1)
print(f"date_threshold : {date_threshold}")
# Create directory if it doesn't exist
save_dir = 'model/data'
os.makedirs(save_dir, exist_ok=True)
filename = "model/data/combined_output.csv"
filepath = "model/data"
print(f"Filename to read : {filename}")

date_threshold : 2024-12-01 00:00:00
Filename to read : model/data/combined_output.csv


## I. PCA Analysis

### 1. Fetch data for the FAANG stocks plus additional promising stocks

In [29]:
tablename = "stocksinfp"
stock_change_tracker_table = "stock_change_tracker"
# Load configuration details
try:
    url, driver, username, password = load_config()
    print(f"URL: {url}, Driver: {driver}, Username: {username}, Table: {tablename}")
except Exception as e:
    print(f"Error loading config: {e}")
spark = initialize_spark("Stock Market Data Analysis Functions")

URL: jdbc:mysql://localhost/stocksdb, Driver: com.mysql.cj.jdbc.Driver, Username: quizadmin, Table: stocksinfp


## Data Preparation as Yahoo

In [30]:
stock_df = spark.read.format("jdbc").options(url=url, driver=driver, user=username, password=password, dbtable=tablename).load()
change_tracker_df = spark.read.format("jdbc").options(url=url, driver=driver, user=username, password=password, dbtable=stock_change_tracker_table).load()


In [31]:
stock_df = stock_df.filter(col("Date") >= date_threshold).select('Date','Symbol','Open', 'High', 'Low', 'Close', 'Volume')
# Count the number of rows per symbol
symbol_counts = stock_df.groupBy("Symbol").count()
# Filter symbols that have at least 50 rows
valid_symbols = symbol_counts.filter(col("count") >= 50).select("Symbol")
# Join with the original DataFrame to keep only valid symbols
stock_df = stock_df.join(valid_symbols, on="Symbol", how="inner")
stock_df = stock_df.withColumnRenamed("Symbol", "Ticker").withColumn("Volume",col("Volume").cast(IntegerType()))
stock_df = stock_df.fillna({'Close': 0, 'High': 0, 'Low': 0, 'Open': 0, 'Volume': 0})
unique_symbol_df = stock_df.select("Ticker").distinct()
#stock_df.show(5, truncate=False)

In [32]:
# Filter data for positive earnings or NVDA
filtered_df = change_tracker_df.filter(
    ((col("change_since_added") > 1) & (col("change_since_added").between(20, 60))) | 
    (col("Symbol").isin("AAPL", "AMZN", "NVDA"))
)
# Get top 50 by change_since_added
top_50_df = filtered_df.join(unique_symbol_df, filtered_df["Symbol"] == unique_symbol_df["Ticker"], "inner").select("Symbol", "change_since_added") \
                        .orderBy(desc("change_since_added")) \
                        .limit(20)
#top_50_df.show(5, truncate=False)
# Ensure NVDA is included
nvda_df = filtered_df.filter(col("Symbol").isin("AAPL", "AMZN", "NVDA")).select("Symbol", "change_since_added")

# Union NVDA if it was not in the top 50
change_tracker_final_df = top_50_df.union(nvda_df).distinct()
#change_tracker_final_df = nvda_df

# Show NVDA results separately
#final_df.filter(col("Symbol") == "NVDA").show(10, truncate=False)

# Show final DataFrame
# change_tracker_final_df.show(5, truncate=False)


In [33]:
best_50_df = stock_df.join(change_tracker_final_df, stock_df["Ticker"] == change_tracker_final_df["Symbol"], "inner").drop("change_since_added","Symbol")
best_50_df.show(5, truncate=False)
tickers = best_50_df.select("Ticker").distinct().rdd.flatMap(lambda x: x).collect()
best_50_df.printSchema()

+------+----------+------------------+------------------+-----------------+-----------------+------+
|Ticker|Date      |Open              |High              |Low              |Close            |Volume|
+------+----------+------------------+------------------+-----------------+-----------------+------+
|ROOT  |2024-12-02|100.66999816894531|102.4800033569336 |95.43800354003906|99.2699966430664 |411300|
|ROOT  |2024-12-03|99.69999694824219 |103.90799713134766|91.9800033569336 |92.86000061035156|369100|
|ROOT  |2024-12-04|92.72000122070312 |94.72000122070312 |89.01000213623047|90.44999694824219|414900|
|ROOT  |2024-12-05|91.26000213623047 |97.62000274658203 |90.2699966430664 |95.70999908447266|379500|
|ROOT  |2024-12-06|95.41999816894531 |98.36000061035156 |92.80899810791016|97.68000030517578|334600|
+------+----------+------------------+------------------+-----------------+-----------------+------+
only showing top 5 rows



root
 |-- Ticker: string (nullable = true)
 |-- Date: date (nullable = true)
 |-- Open: string (nullable = false)
 |-- High: string (nullable = false)
 |-- Low: string (nullable = false)
 |-- Close: string (nullable = false)
 |-- Volume: integer (nullable = false)



In [34]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

import os
import glob

def combine_csv_files(output_path, final_filename="combined_output.csv"):
    """
    Combines the header.csv and data part-file into a single CSV file
    
    Parameters:
    output_path: Directory containing the header.csv and data files
    final_filename: Name of the final combined file
    """
    # Path for the final combined file
    final_path = os.path.join(output_path, final_filename)
    
    # Read header file
    header_path = os.path.join(output_path, "header.csv")
    
    # Find the data file (it will be named something like part-00000-*.csv)
    data_path = glob.glob(os.path.join(output_path, "data/part-00000-*.csv"))[0]
    
    # Combine the files
    with open(final_path, 'w') as outfile:
        # Write header
        with open(header_path, 'r') as header_file:
            outfile.write(header_file.read())
        
        # Write data
        with open(data_path, 'r') as data_file:
            outfile.write(data_file.read())
    
    # Clean up temporary files
    os.remove(header_path)
    os.system(f"rm -rf {os.path.join(output_path, 'data')}")
    
    print(f"Files combined successfully. Output saved to: {final_path}")

def export_to_yfinance_format(spark_df, output_path, final_filename="combined_output.csv"):
    """
    Export Spark DataFrame to CSV in YFinance format with multiple tickers
    
    Parameters:
    spark_df: Spark DataFrame with stock data containing Date, Ticker, Open, High, Low, Close, Volume
    output_path: String path where to save the CSV file
    """
    # Get unique tickers first
    tickers_df = spark_df.select("Ticker").distinct()
    ticker_list = [row.Ticker for row in tickers_df.collect()]
    
    # Create the header strings
    ticker_row = "Ticker," + ",".join([ticker for ticker in ticker_list for _ in range(5)])
    price_row = "Price," + ",".join(["Open,High,Low,Close,Volume"] * len(ticker_list))
    date_row = "Date,," + ",".join([",,,," for _ in range(len(ticker_list))])
    
    # Format the data
    formatted_df = spark_df.withColumn(
        "Date",
        F.date_format("Date", "yyyy-MM-dd")
    )
    
    # First, create separate columns for each metric and ticker
    final_df = formatted_df.select("Date")
    
    for ticker in ticker_list:
        ticker_data = formatted_df.filter(F.col("Ticker") == ticker)
        
        # Create columns for this ticker
        ticker_df = ticker_data.select(
            "Date",
            F.col("Open").alias(f"{ticker}_Open"),
            F.col("High").alias(f"{ticker}_High"),
            F.col("Low").alias(f"{ticker}_Low"),
            F.col("Close").alias(f"{ticker}_Close"),
            F.col("Volume").alias(f"{ticker}_Volume")
        )
        
        # Join with the final DataFrame
        final_df = final_df.join(ticker_df, "Date", "outer")
    
    # Order columns correctly
    ordered_cols = ["Date"]
    for ticker in ticker_list:
        ordered_cols.extend([
            f"{ticker}_Open", f"{ticker}_High", f"{ticker}_Low", 
            f"{ticker}_Close", f"{ticker}_Volume"
        ])
    
    final_df = final_df.select(ordered_cols).orderBy("Date")
    # Write the headers to a file
    with open(f"{output_path}/header.csv", "w") as f:
        f.write(f"{ticker_row}\n")
        f.write(f"{price_row}\n")
        f.write(f"{date_row}\n")
    # Write the data
    final_df.coalesce(1).write.mode("overwrite").option("header", "false").option("sep", ",").csv(f"{output_path}/data")
    # Combine the files
    combine_csv_files(output_path, final_filename)
    
    # Note: You'll need to concatenate the header.csv with the generated data file
    # using system commands or file operations

export_to_yfinance_format(best_50_df, filepath)   

25/02/23 21:22:51 WARN DAGScheduler: Broadcasting large task binary with size 1178.8 KiB


Files combined successfully. Output saved to: model/data/combined_output.csv


In [35]:
print("completed")
    
    

completed
